In [ ]:
import liana as li
import omnipath as op
import decoupler as dc
import scanpy as sc
import pandas as pd
import numpy as np

### loading data

In [ ]:
adata_t = sc.read_h5ad("/content/xenium/validation/atlas_tumor_only.h5ad")

In [ ]:
adata_t = adata_t[adata_t.obs['atlas_cell_type_coarse'].isin(["T cell", "NK cell", "ILC", "Cancer cell", "Epithelial cell"])].copy()

In [ ]:
adata_t.var_names = adata_t.var["var_names"]

In [ ]:
# Import T cell annotation
train = pd.read_csv("/content/drive/MyDrive/xenium/validation/snhx_meta.csv", index_col=0)

In [ ]:
adata_t.obs['cell_type_updated'] = adata_t.obs['atlas_cell_type_fine'].astype(str)
adata_t.obs['cell_type_updated'].update(train['t_sub_3'])

In [ ]:
adata_t = adata_t[adata_t.obs['microsatellite_status'].isin(["MSI", "MSI-H", "MSS"])].copy()

In [ ]:
adata_t.layers["data"] = adata_t.X

In [ ]:
adata_t.X = adata_t.layers["data"]

In [ ]:
# Log-normalize raw cell counts
sc.pp.normalize_total(adata_t)
sc.pp.log1p(adata_t)

In [ ]:
ligrec = op.interactions.import_intercell_network()

In [ ]:
ligrec = ligrec.rename(columns={'genesymbol_intercell_source':'ligand', 'genesymbol_intercell_target':'receptor'})
ligrec = ligrec[['ligand', 'receptor', 'references'] + [col for col in ligrec.columns if col not in ['ligand', 'receptor', 'references']]]
ligrec = ligrec[ligrec["references"].str.contains("CellPhoneDB|CellChatDB|connectomeDB2020", na=False)]
ligrec = ligrec[ligrec["consensus_direction"] == True]

In [ ]:
adata_MSI = adata_t[adata_t.obs['microsatellite_status'].isin(["MSI", "MSI-H"])].copy()
adata_MSS = adata_t[adata_t.obs['microsatellite_status'].isin(["MSS"])].copy()

### Cell to cell

In [ ]:
# Run rank_aggregate
li.mt.rank_aggregate(adata_MSS,
                     groupby='cell_type_updated',
                     resource = ligrec[["ligand", "receptor"]],
                     n_jobs = 5,
                     expr_prop=0.05,
                     verbose=True,
                     use_raw = False)
res = adata_MSS.uns['liana_res']

In [ ]:
MSI_liana_res_filtered = adata_MSI.uns['liana_res'].copy()
MSI_liana_res_filtered = MSI_liana_res_filtered[
    (MSI_liana_res_filtered["specificity_rank"] < 0.05) &
    (MSI_liana_res_filtered['ligand_complex'] == 'IL15') &
    (MSI_liana_res_filtered['receptor_complex'] == 'IL2RB')
]

In [ ]:
MSS_liana_res_filtered = adata_MSS.uns['liana_res'].copy()
MSS_liana_res_filtered = MSS_liana_res_filtered[
    (MSS_liana_res_filtered["specificity_rank"] < 0.05) &
    (MSS_liana_res_filtered['ligand_complex'] == 'IL15') &
    (MSS_liana_res_filtered['receptor_complex'] == 'IL2RB')
]

In [ ]:
MSI_liana_res_filtered
MSS_liana_res_filtered